In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Employee salary records

The HR team exported employee records before a headcount audit. The data has four problems:

1. **Duplicates** — some employees were exported more than once. Remove exact duplicate rows.
2. **`department`** — values like `"  Engineering"`, `"MARKETING"`, `"sales"`. Standardize to lowercase with no leading/trailing whitespace.
3. **`salary`** — stored as strings like `"$92,500"`. Strip the `$` and `,`, then convert to float.
4. **`hire_date`** — some entries are `None`. Fill them with `"2020-01-01"`.

**New this week:**

`drop_duplicates()` — removes rows that are identical across all columns. Returns a new DataFrame (original untouched). To check only specific columns: `df.drop_duplicates(subset=['col1', 'col2'])`.

`pd.to_numeric(series, errors='coerce')` — converts a Series to float. Values that can't be parsed (like text or `"n/a"`) become `NaN` instead of raising an error. Use it after cleaning the string format.

`fillna(value)` — replaces `NaN` with `value`. For forward-fill: `fillna(method='ffill')`.

After cleaning, print:
- How many rows were removed as duplicates
- Mean salary per department, rounded to the nearest dollar
- Which department has the highest average salary

In [20]:
employees = pd.DataFrame({
    'employee_id': [101, 102, 103, 104, 105, 103, 106, 107, 102],
    'name':        ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Carol', 'Frank', 'Grace', 'Bob'],
    'department':  ['  Engineering', 'Marketing', 'engineering', 'Sales', 'MARKETING',
                    'engineering',   'sales',      'Engineering', 'Marketing'],
    'salary':      ['$92,500', '$64,000', '$87,300', '$71,200', '$68,500',
                    '$87,300', '$73,100', '$95,000', '$64,000'],
    'hire_date':   ['2019-03-15', '2020-07-22', '2018-11-01', None, '2021-02-14',
                    '2018-11-01', None, '2017-06-30', '2020-07-22'],
})

# Your code here

new_employees = employees.drop_duplicates().copy()
new_employees['salary'] = new_employees['salary'].str.replace('$','')
new_employees['salary'] = new_employees['salary'].str.replace(',','')
new_employees['salary'] = pd.to_numeric(new_employees['salary'], errors = 'coerce')
new_employees['department'] = new_employees['department'].str.lower()
new_employees['department'] = new_employees['department'].str.replace(' ','')
new_employees['hire_date'] = new_employees['hire_date'].fillna('2020-01-01')


print(employees.shape[0] - new_employees.shape[0], 'are duplicated rows')
print(new_employees.groupby('department')['salary'].apply(lambda x: x.mean().round()))
print(new_employees.groupby('department')['salary'].apply(lambda x: x.mean().round()).idxmax(),'has the highest average salary')


2 are duplicated rows
department
engineering    91600.0
marketing      66250.0
sales          72150.0
Name: salary, dtype: float64
engineering has the highest average salary


---

## Level 2 — Product catalog

A product catalog scraped from a retailer's website. The `price` column is a mess — it has values like `"999.99"`, `"FREE"`, `"€79.99"`, and `"n/a"`. The `category` column has inconsistent casing and whitespace. One product has both `price` and `rating` missing — drop it.

Clean the data, then answer these questions:

1. What is the median price per category? Use `np.nanmedian` per group.
2. What fraction of products are rated 4.0 or above? (`np.nanmean` on a boolean comparison — treat missing ratings as below threshold)
3. How many products are in each category? List from most to fewest.

**Price cleaning:** replace `"FREE"` with `"0"` using `.replace()`, strip `"€"` with `.str.replace()`, then `pd.to_numeric(errors='coerce')` converts the rest (turning `"n/a"` into NaN).

**New tool — `np.clip(array, a_min, a_max)`:** caps values at bounds. After converting prices, one product has a price that's clearly wrong. Find the 95th percentile with `np.nanpercentile`, then cap:
```python
cap = np.nanpercentile(catalog['price'].dropna(), 95)
catalog['price'] = np.clip(catalog['price'], 0, cap)
```

In [49]:
catalog = pd.DataFrame({
    'product':  ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Headphones',
                 'Webcam', 'USB Hub', 'Cable', 'Charger', 'Desk Lamp',
                 'Speaker', 'Tablet', 'Stand', 'Printer', 'Router'],
    'category': ['  Electronics', 'electronics', 'ELECTRONICS', 'Electronics', 'electronics ',
                 'Accessories ', 'accessories', 'ACCESSORIES', 'Accessories', 'Accessories',
                 ' Electronics', 'ELECTRONICS', 'accessories', 'Peripherals', 'Peripherals'],
    'price':    ['999.99', '25.50', 'FREE', '349.00', '€79.99',
                 '89.00', '35.00', 'n/a', '29.99', '45.00',
                 '149.99', '549.00', '79.00', '299.00', '129.99'],
    'rating':   [4.5, 4.2, 4.8, 4.6, 4.1,
                 3.9, 4.1, None, 4.3, 4.0,
                 4.4, 4.5, 4.2, None, 4.3],
    'stock':    [15, 200, 150, 8, 45,
                 30, 75, None, 120, 60,
                 25, 5, 40, 12, 20],
})

# Your code here

catalog['price'] = catalog['price'].str.replace('FREE', '0')
catalog['price'] = catalog['price'].str.replace('€', '')
catalog['price'] = pd.to_numeric(catalog['price'], errors= 'coerce')

catalog['category'] = catalog['category'].str.lower().str.replace(r'\s+','', regex = True)

newcat = catalog.dropna(subset=['price', 'rating'],how = 'all').copy()
cap = np.nanpercentile(newcat['price'].dropna(),95)
newcat['price'] = np.clip(newcat['price'],0, cap)

print('new_catalog median per cat is: ', newcat.groupby('category')['price'].apply(lambda x: np.nanmean(x)))

print(np.nanmean(newcat['rating']>=4.0), 'are rated 4.0 and above')
gc = newcat.groupby('category')['product'].agg('count')
aa = np.argsort(-gc)
print(gc.iloc[aa])



new_catalog median per cat is:  category
accessories     55.598000
electronics    265.760929
peripherals    214.495000
Name: price, dtype: float64
0.8571428571428571 are rated 4.0 and above
category
electronics    7
accessories    5
peripherals    2
Name: product, dtype: int64


---

## Level 3 — Food distributor orders

A regional food distributor exported 16 orders. The file has a duplicated order, region names with inconsistent casing and whitespace, two `"unknown"` quantities, and one missing unit price.

**Step 1 — Build a `.pipe()` cleaning pipeline.** Write three functions and chain them:

```python
def clean_strings(df):
    # standardize region: lowercase, strip whitespace
    ...

def fix_types(df):
    # convert quantity and unit_price to numeric (errors='coerce')
    ...

def drop_bad_rows(df):
    # drop rows where quantity or unit_price is NaN
    # then drop duplicate rows
    ...

clean = orders.pipe(clean_strings).pipe(fix_types).pipe(drop_bad_rows).copy()
```

**Step 2 — Revenue analysis:**

1. Add `revenue = clean['quantity'] * clean['unit_price']`
2. Convert `order_date` to datetime. Group by month (`.dt.to_period('M')`) and sum revenue to get a monthly Series.
3. On that monthly Series: compute `ewm_rev = series.ewm(span=3).mean()` and `cum_avg = series.expanding().mean()`. At the final month, is revenue accelerating or decelerating?
4. Which region has the highest total revenue across all cleaned orders? Use `np.argmax` on the grouped result.

In [74]:
orders = pd.DataFrame({
    'order_id':   ['O001', 'O002', 'O003', 'O004', 'O005', 'O006', 'O007', 'O008',
                   'O009', 'O010', 'O011', 'O012', 'O013', 'O014', 'O015', 'O008'],
    'region':     [' north', 'SOUTH', 'East', 'west ', 'NORTH', 'south ', 'EAST', 'West',
                   'north', ' SOUTH', 'east', 'West', 'NORTH', 'south', 'EAST ', 'West'],
    'quantity':   [120, 85, 200, 150, 95, 'unknown', 180, 210,
                   130, 165, 90, 250, 140, 175, 'unknown', 210],
    'unit_price': [2.50, 3.20, 1.80, 2.50, 3.20, 1.80, None, 2.50,
                   3.20, 2.50, 1.80, 3.20, 2.50, 1.80, 3.20, 2.50],
    'order_date': ['2024-01-10', '2024-01-25', '2024-02-08', '2024-02-20', '2024-03-05',
                   '2024-03-18', '2024-03-25', '2024-04-02', '2024-04-14', '2024-04-25',
                   '2024-05-09', '2024-05-22', '2024-06-03', '2024-06-17', '2024-06-28',
                   '2024-04-02'],
})

# Your code here


def clean_strings(df):
    df['region'] = df['region'].str.lower().str.replace(r'\s+','',regex = True)
    return df
def fix_types(df):
    df['quantity'] = pd.to_numeric(df['quantity'], errors = 'coerce')
    df['unit_price'] = pd.to_numeric(df['unit_price'], errors = 'coerce')
    return df 
def drop_bad_rows(df):
    df = df.dropna(subset = ['quantity','unit_price'])
    df = df.drop_duplicates()
    return df

clean = orders.pipe(clean_strings).pipe(fix_types).pipe(drop_bad_rows).copy()
clean['revenue'] = clean['quantity'] * clean['unit_price']
clean['order_date'] = pd.to_datetime(clean['order_date'])
clean['month'] = clean['order_date'].dt.to_period('M')
gs = clean.groupby('month')['revenue'].sum()
print(gs)
gs_df = pd.DataFrame(gs)
gs_df['ewm_rev'] = gs_df['revenue'].ewm(span = 3).mean()
gs_df['cum_rev'] = gs_df['revenue'].expanding().mean()
gg = gs_df['ewm_rev']- gs_df['cum_rev']
if gg.iloc[-1]>0:
    print('revenue is accelerating')
else:
    print('revenue is decelerating')

gr = clean.groupby('region')['revenue'].sum()
print(gr.index[np.argmax(gr)],'had the highest total rev')

month
2024-01     572.0
2024-02     735.0
2024-03     304.0
2024-04    1353.5
2024-05     962.0
2024-06     665.0
Freq: M, Name: revenue, dtype: float64
revenue is accelerating
west had the highest total rev


In [70]:
gg.iloc[-1]

np.float64(40.43253968253964)